# Figure 5: maximum-likelihood whitened waveforms

Input: `data/fig05_whitened_waveforms.npz`, containing the plotted arrays generated from public GWOSC O4b strain. The exact maximum-likelihood REMOVEQ5 parameter point is retained below instead of the full posterior. Time is in GPS seconds and the plotted quantity is whitened strain. In the parameter point, `srcmchirp` is in solar masses, `comoving_volume` in Mpc$^3$, angular variables in radians, and all other quantities are dimensionless.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    raise RuntimeError('Run this notebook from the repository root.')

fig_width = 3 * 246.0 / 72.27
fig_height = fig_width * (np.sqrt(5) - 1.0) / 2.0
mpl.rcParams.update({
    'axes.labelsize': 24,
    'font.family': 'serif',
    'font.serif': 'Computer Modern Roman',
    'font.size': 24,
    'legend.fontsize': 20,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'axes.grid': True,
    'text.usetex': True,
    'savefig.dpi': 100,
    'lines.markersize': 14,
    'figure.figsize': [fig_width, fig_height],
})

maximum_likelihood_sample = {
    'srcmchirp': 28.751638203029206,
    'q': 1.0039733701041689,
    'spin1_a': 0.07511133279642021,
    'spin1_azimuthal': 0.6703529528240063,
    'spin1_polar': 2.118775970725862,
    'spin2_a': 0.18260998822863586,
    'spin2_azimuthal': 0.8222346938434326,
    'spin2_polar': 1.625286951664129,
    'comoving_volume': 207331945.36931118,
    'coa_phase': 1.377266746476411,
    'inclination': 0.8205702205313092,
    'loglikelihood': -1416683.3529836314,
}
original_sample_index = 755268

data = np.load(ROOT / 'data/fig05_whitened_waveforms.npz')
wfactor = float(data['wfactor'])
assert original_sample_index == int(data['maxl_index'])
assert maximum_likelihood_sample['loglikelihood'] == float(data['maxl_loglikelihood'])
print(f"maximum-likelihood sample index = {original_sample_index}")
print(f"maximum log likelihood = {maximum_likelihood_sample['loglikelihood']:.12f}")
print(f"toffset = {float(data['toffset']):.8f} s")
maximum_likelihood_sample

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True)
for axis, detector, label in (
    (axes[0], 'H1', 'LIGO Hanford data'),
    (axes[1], 'L1', 'LIGO Livingston data'),
):
    merger_time = float(data[f'{detector}_merger_time'])
    axis.plot(data[f'{detector}_data_t'], data[f'{detector}_data_h'] / wfactor,
              color='grey', alpha=0.5, label=label)
    axis.plot(data[f'{detector}_imr_t'], data[f'{detector}_imr_h'] / wfactor,
              label='Inspiral-Merger: NRSur7dq4' if detector == 'H1' else None)
    axis.plot(data[f'{detector}_ringdown_t'], data[f'{detector}_ringdown_h'] / wfactor,
              label='Ringdown: All Modes' if detector == 'H1' else None)
    axis.plot(data[f'{detector}_qqnm_t'], data[f'{detector}_qqnm_h'] / wfactor,
              label='Ringdown: Quadratic Modes' if detector == 'H1' else None)
    axis.axvline(merger_time, color='k', linestyle='--', alpha=0.8,
                 label='merger time' if detector == 'L1' else None)
    axis.set_ylim(-420 / wfactor, 420 / wfactor)
    axis.set_xlim(merger_time - 0.1, merger_time + 0.03)
    axis.legend(fontsize=12, ncols=2)

axes[1].set_xlabel('Time (s)')
fig.text(0.02, 0.5, 'Whitened Strain', va='center', rotation='vertical', fontsize=22)
plt.subplots_adjust(hspace=0)
output = ROOT / 'figures/figure5_maxl_4Mf.pdf'
fig.savefig(output, bbox_inches='tight')
plt.show()